In [3]:
import os
#1.导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI
import dotenv
from langchain_classic.chains.llm import LLMChain
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ChatMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

chat_model = ChatOpenAI(
    model='qwen-plus',
    api_key=os.getenv('ALI_API_KEY'),
    base_url=os.getenv('ALI_API_URL'),
)

prompt_template = PromptTemplate(template="你是一个{role},你的名字叫{name}",input_variables=["role", "name"])

fmt = prompt_template.format(role="有帮助的助手", name="AIBOT")
print(fmt)

# 3.定义工具
tools = [MoveFileTool()]
# 4.这里需要将工具转换为openai函数，后续再将函数传入模型调用
functions = [convert_to_openai_function(t) for t in tools]
# print(functions[0])
# 5. 提供大模型调用的消息列表
messages = [HumanMessage(content="将文件a移动到桌面")]
# 6.模型使用函数
response = chat_model.invoke(
input = messages,
functions=functions
)
print(response)

print("今天的天气怎么样？ 的回复内容 ==============================")
response = chat_model.invoke(
[HumanMessage(content="今天的天气怎么样？")],
functions=functions
)
print(response)

你是一个有帮助的助手,你的名字叫AIBOT
content='' additional_kwargs={'function_call': {'arguments': '{"source_path": "a", "destination_path": "~/Desktop/a"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 194, 'total_tokens': 222, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-36574287-81d6-91e5-894a-f43e3d8bd480', 'finish_reason': 'function_call', 'logprobs': None} id='lc_run--019cf0d4-ea41-75c2-b7ef-3ca6f510a5b1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 194, 'output_tokens': 28, 'total_tokens': 222, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
今天的天气怎么样？ 的回复内容 ==============================
content='我无法提供实时天气信息。建议您查看当地的天气预报应用或网站，如中国气象局官网、天气通等，以获取最新天气情况。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'

In [4]:
from langchain_core.messages import AIMessage

msg = AIMessage(
content='', # 无自然语言回复
additional_kwargs={
'function_call': {
'name': 'move_file', # 工具名称
'arguments':
'{"source_path":"a","destination_path":"/Users/YourUsername/Desktop/a"}' # 工具参数
}
}
)
print(msg)


content='' additional_kwargs={'function_call': {'name': 'move_file', 'arguments': '{"source_path":"a","destination_path":"/Users/YourUsername/Desktop/a"}'}} response_metadata={} tool_calls=[] invalid_tool_calls=[]


In [8]:
# 定义LLM模型

# 定义工具
tools = [MoveFileTool()]
# 将工具转换为openai函数
functions = [convert_to_openai_function(t) for t in tools]
# 提供消息列表
messages = [HumanMessage(content="将本目录下的abc.txt文件移动到 C:\\Users\\oo\\Desktop\\code\\python\\langchain_demo\\sgg\\")]
# 模型调用
response = chat_model.invoke(
input=messages,
functions=functions
)
print(response)

import json
if "function_call" in response.additional_kwargs:
    tool_name = response.additional_kwargs["function_call"]["name"]
    tool_args = json.loads(response.additional_kwargs["function_call"]["arguments"])
    print(f"调用工具: {tool_name}, 参数: {tool_args}")
else:
    print("模型回复:", response.content)

if "move_file" in response.additional_kwargs["function_call"]["name"]:
    tool = MoveFileTool()
    result = tool.run(tool_args) # 执行工具
    print("工具执行结果:", result)


content='' additional_kwargs={'function_call': {'arguments': '{"source_path": "abc.txt", "destination_path": "C:\\\\Users\\\\oo\\\\Desktop\\\\code\\\\python\\\\langchain_demo\\\\sgg\\\\"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 215, 'total_tokens': 261, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-70ac1505-4f28-996e-9378-15a419f3324a', 'finish_reason': 'function_call', 'logprobs': None} id='lc_run--019cf106-6335-7732-82f5-1803800d8d23-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 215, 'output_tokens': 46, 'total_tokens': 261, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
调用工具: move_file, 参数: {'source_path': 'abc.txt', 'destination_path': 'C:\\Users\\oo\\Desktop\\code\\python\\langchain_demo\\sgg\\'}
工具执行结